# Fase 6 (b) — Estudio de ablación: función de pérdida y conexiones de salto  (Colab)

**Entregable 2(b):** comparar BCE vs. Dice vs. combinada, y una decisión arquitectónica (quitar las conexiones de salto).

| Variante | Pérdida | Saltos | Pregunta que responde |
|---|---|---|---|
| `bce` | BCE | sí | referencia (= base de la fase 5) |
| `dice` | Dice | sí | ¿una pérdida centrada en la clase agua sube el **recall** (baja la omisión)? |
| `combinada` | 0.5 BCE + 0.5 Dice | sí | ¿se obtiene lo mejor de ambas? |
| `sin_saltos` | BCE | **no** | ¿cuánto aportan los saltos al detalle fino (ríos angostos, bordes)? |

**Rigor experimental:**
- **3 semillas por variante** (42, 7, 123): cambia la inicialización, el orden de los lotes y los aumentos. En la fase 5 vimos que el IoU de valid oscila ±0.03 entre épocas → con una sola corrida no se puede distinguir una mejora real de ruido.
- **Mismo presupuesto para todas:** 60 épocas máx., paciencia 15, misma tasa, mismos datos.
- **Todo se compara en validación.** Test sigue intacto hasta la fase 7.
- **Lectura de resultados:** si la diferencia de medias entre dos variantes es menor que sus desviaciones, **no** hay evidencia de que una sea mejor.
- **Reanudable:** cada corrida terminada se salta al re-ejecutar (si la sesión se cae, solo corre lo que falta).

**Colab:** requiere GPU T4. Tiempo estimado **~1 hora**. Si la sesión se cae, vuelve a *Ejecutar todo*: las corridas terminadas se reutilizan.
Usa los módulos de las fases anteriores que ya están en tu Drive; solo escribe los **2 nuevos**.

## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
sys.path.insert(0, str(RAIZ_PROYECTO))

!pip install -q rasterio
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Módulos nuevos de `src/`

In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/ablacion.py
"""
ablacion.py
Corre una lista de configuraciones x semillas, evalúa cada mejor modelo en validación y resume.
- Cada corrida se guarda en resultados/experimentos/<nombre>_s<semilla>/
- Si una corrida ya terminó (existe resumen.json), se SALTA -> si la sesión se cae, al volver
  a ejecutar solo se corre lo que falta.
- Cargadores nuevos por semilla: así el orden de los lotes y los aumentos también dependen de
  la semilla (variabilidad completa, no solo la de la inicialización de pesos).
"""
import gc
import json
from dataclasses import replace

import pandas as pd
import torch

from .cargadores import crear_cargadores
from .entrenamiento import entrenar, cargar_mejor_modelo
from .evaluacion import evaluar
from .utilidades import fijar_semilla


def correr_ablacion(variantes: dict, semillas, normalizador, dispositivo, dir_experimentos,
                    tamano_lote: int = 8, tamano_recorte: int = 256, num_workers: int = 2) -> pd.DataFrame:
    """variantes: {"nombre_variante": ConfiguracionExperimento, ...}"""
    filas = []
    for nombre_variante, configuracion_base in variantes.items():
        for semilla in semillas:
            configuracion = replace(configuracion_base, nombre=f"{nombre_variante}_s{semilla}", semilla=semilla)
            dir_corrida = dir_experimentos / configuracion.nombre
            print(f"\n{'=' * 70}\n▶ {configuracion.nombre}\n{'=' * 70}")

            fijar_semilla(semilla)
            cargadores = crear_cargadores(normalizador, tamano_lote, tamano_recorte, num_workers, semilla)
            if not (dir_corrida / "resumen.json").exists():
                entrenar(configuracion, cargadores["train"], cargadores["valid"], dispositivo, dir_corrida)
                (dir_corrida / "ultimo.pt").unlink(missing_ok=True)   # solo servía para reanudar
            else:
                print("  ya estaba terminada: se reutiliza")

            modelo, _, mejor_epoca = cargar_mejor_modelo(dir_corrida, dispositivo)
            acumulador, _ = evaluar(modelo, cargadores["valid"], dispositivo)
            acumulador.tabla_por_evento().to_csv(dir_corrida / "por_evento_valid.csv")
            with open(dir_corrida / "metricas_valid.json", "w") as f:
                json.dump(acumulador.metricas(), f, indent=2)
            filas.append({"variante": nombre_variante, "semilla": semilla, "mejor_epoca": mejor_epoca,
                          **acumulador.metricas()})

            del modelo, cargadores
            gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(filas)


def resumir(df: pd.DataFrame, metricas=("iou", "f1", "precision", "recall", "omision", "comision")) -> pd.DataFrame:
    """Media ± desviación estándar por variante (sobre las semillas)."""
    agrupado = df.groupby("variante", sort=False)[list(metricas)]
    media, desviacion = agrupado.mean(), agrupado.std(ddof=1)
    texto = media.round(3).astype(str) + " ± " + desviacion.round(3).astype(str)
    texto.insert(0, "n_semillas", df.groupby("variante", sort=False).size())
    return texto


def por_evento_promedio(nombres_variantes, semillas, dir_experimentos) -> pd.DataFrame:
    """IoU por evento promediado sobre semillas (filas = evento, columnas = variante)."""
    columnas = {}
    for variante in nombres_variantes:
        tablas = [pd.read_csv(dir_experimentos / f"{variante}_s{s}" / "por_evento_valid.csv", index_col=0)["iou"]
                  for s in semillas]
        columnas[variante] = pd.concat(tablas, axis=1).mean(axis=1)
    return pd.DataFrame(columnas).sort_values(list(columnas)[0])


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/visualizacion_ablacion.py
"""
visualizacion_ablacion.py
Figuras del estudio de ablación (entregable 2).
"""
import matplotlib.pyplot as plt
import numpy as np


def figura_ablacion(df, linea_base_iou=None, titulo="", ruta_guardado=None):
    """Izquierda: IoU medio ± desv. por variante (puntos = cada semilla).
    Derecha: precisión vs recall de cada corrida (muestra el compromiso omisión/comisión)."""
    variantes = list(dict.fromkeys(df["variante"]))
    colores = plt.cm.tab10(np.arange(len(variantes)))
    figura, ejes = plt.subplots(1, 2, figsize=(13, 4.2))

    for i, (variante, color) in enumerate(zip(variantes, colores)):
        valores = df.loc[df["variante"] == variante, "iou"]
        ejes[0].bar(i, valores.mean(), yerr=valores.std(ddof=1) if len(valores) > 1 else 0,
                    color=color, alpha=0.75, capsize=6)
        ejes[0].scatter(np.full(len(valores), i), valores, color="black", s=14, zorder=3)
        ejes[0].text(i, valores.mean() + 0.01, f"{valores.mean():.3f}", ha="center", fontsize=9)
        sub = df[df["variante"] == variante]
        ejes[1].scatter(sub["recall"], sub["precision"], color=color, label=variante, s=40)
    if linea_base_iou is not None:
        ejes[0].axhline(linea_base_iou, ls="--", color="gray", label=f"Otsu = {linea_base_iou:.3f}")
        ejes[0].legend()
    ejes[0].set_xticks(range(len(variantes)), variantes)
    minimo = min(df["iou"].min(), linea_base_iou or 1) - 0.05
    ejes[0].set_ylim(max(0, minimo), min(1, df["iou"].max() + 0.05))
    ejes[0].set_ylabel("IoU agua (validación)")
    ejes[0].set_title("IoU por variante (media ± desv., puntos = semillas)")
    ejes[1].set_xlabel("Recall (1 − omisión)"); ejes[1].set_ylabel("Precisión (1 − comisión)")
    ejes[1].set_title("Compromiso precisión / recall"); ejes[1].legend()
    for eje in ejes:
        eje.grid(alpha=0.3)
    figura.suptitle(titulo)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


## 2. (Colab ya tiene en Drive los archivos de fases anteriores)

In [ ]:
import pandas as pd
from src.configuracion import *

## 3. Imports, datos y normalizador

In [ ]:
import torch
from src.utilidades import obtener_dispositivo
from src.descarga import descargar_mano
from src.preprocesamiento import Normalizador
from src.experimento import ConfiguracionExperimento
from src.ablacion import correr_ablacion, resumir, por_evento_promedio
from src.visualizacion_ablacion import figura_ablacion

dispositivo = obtener_dispositivo()
assert dispositivo.type == 'cuda', 'Esta fase necesita GPU (12 corridas)'
descargar_mano()
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')
linea_base_otsu = pd.read_csv(DIR_RESULTADOS / 'f4_linea_base_otsu.csv', index_col=0)

## 4. Variantes y corrida de la ablación

In [ ]:
from dataclasses import replace

base = ConfiguracionExperimento(epocas=60, paciencia=15)        # mismo presupuesto que la fase 5
variantes = {
    'bce':        replace(base, perdida='bce'),
    'dice':       replace(base, perdida='dice'),
    'combinada':  replace(base, perdida='combinada'),
    'sin_saltos': replace(base, perdida='bce', usar_saltos=False),
}
SEMILLAS = [42, 7, 123]
DIR_EXPERIMENTOS = DIR_RESULTADOS / 'experimentos'
print(f"{len(variantes)} variantes x {len(SEMILLAS)} semillas = {len(variantes) * len(SEMILLAS)} corridas")

In [ ]:
df_corridas = correr_ablacion(variantes, SEMILLAS, normalizador, dispositivo, DIR_EXPERIMENTOS,
                              tamano_lote=8, tamano_recorte=256, num_workers=2)
df_corridas.to_csv(DIR_RESULTADOS / 'f6b_ablacion_corridas.csv', index=False)
df_corridas.round(4)

## 5. Resumen: media ± desviación (3 semillas)

In [ ]:
tabla_resumen = resumir(df_corridas)
tabla_resumen.to_csv(DIR_RESULTADOS / 'f6b_ablacion_resumen.csv')
tabla_resumen

## 6. Figura de la ablación

In [ ]:
figura_ablacion(df_corridas, linea_base_otsu.loc['valid', 'iou'],
                'Ablación (b) — validación, 3 semillas', DIR_FIGURAS / 'f6b_ablacion.png');

## 7. IoU por evento (promedio de semillas)

In [ ]:
tabla_eventos = por_evento_promedio(list(variantes), SEMILLAS, DIR_EXPERIMENTOS)
tabla_eventos.to_csv(DIR_RESULTADOS / 'f6b_ablacion_por_evento.csv')
tabla_eventos.round(3)

## Para traer a la conversación
La tabla de corridas (sección 4), el resumen media ± desv. (5), la figura (6) y la tabla por evento (7).
Con eso elegimos la pérdida para la ablación (a) — dorados vs. dorados + débiles.